<a id='functools'></a>

## 14. 🧩 Pattern 14: functools — reduce(), lru_cache, cmp_to_key, partial — LC 70, 139, 179, 198, 322, 494

---

```
PROBLEM:
  LC 70  — Climb Stairs: fib variant — lru_cache turns O(2^n) into O(n)
  LC 139 — Word Break: recursive memoization with lru_cache
  LC 179 — Largest Number: cmp_to_key custom sort comparator
  LC 198 — House Robber: recursive dp + lru_cache
  LC 322 — Coin Change: recursive dp + lru_cache
  LC 494 — Target Sum: recursive dp + lru_cache

reduce(fn, iterable[, initializer]):
  Folds left — applies fn(accumulator, element) across the sequence.
  from functools import reduce
  reduce(lambda a, b: a * b, [1,2,3,4,5])  → 120  (running product)
  reduce(lambda a, b: a + b, [1,2,3], 0)   → 6    (sum with initial value)
  # slow motion on [1,2,3,4]:
  # step 1: fn(1,2) = 2   → acc=2
  # step 2: fn(2,3) = 6   → acc=6
  # step 3: fn(6,4) = 24  → acc=24

@lru_cache — THE Pythonic memoization decorator:
  from functools import lru_cache

  @lru_cache(maxsize=None)   # cache unlimited results
  def fib(n):
      if n <= 1: return n
      return fib(n-1) + fib(n-2)

  lru_cache replaces the manual memo dict pattern:
  # OLD — manual memo
  def fib(n, memo={}):
      if n in memo: return memo[n]
      if n <= 1: return n
      memo[n] = fib(n-1) + fib(n-2)
      return memo[n]

  # NEW — decorator handles it
  @lru_cache(maxsize=None)
  def fib(n): ...

  cache_info() shows hits/misses:
  fib.cache_info()  → CacheInfo(hits=N, misses=M, maxsize=None, currsize=M)

cmp_to_key — custom comparator:
  from functools import cmp_to_key
  def compare(a, b):   # returns negative/zero/positive (C-style)
      ...
  sorted(items, key=cmp_to_key(compare))

partial — pre-fill arguments:
  from functools import partial
  double = partial(pow, exp=2)    ← fixes exp=2, first arg still free
  add5   = partial(lambda x,y: x+y, 5)

KEY INSIGHT:
  lru_cache is @memoize for free — one decorator line converts any
  recursive solution from exponential to linear. Use it first in interviews.

TIME / SPACE:
  lru_cache: Time O(n) — each unique arg computed once; Space O(n) — cache stores results
  reduce:    Time O(n) — one pass; Space O(1) — accumulator only
```

In [ ]:
from functools import reduce, lru_cache, cmp_to_key, partial
from typing import List


# ── reduce() ─────────────────────────────────────────────────────────────────
# Think: fold-left — "carry the result forward and smash the next item into it"

def product_of_array(nums: List[int]) -> int:
    """
    LC 1570-variant — product of all elements using reduce.
    Approach: fold-left with multiplication operator.
    Args:
        nums (List[int]): non-empty list of integers.
    Returns:
        int: product of all elements.
    Time:  O(n) — one pass, one multiply per element
    Space: O(1) — accumulator only
    """
    return reduce(lambda acc, x: acc * x, nums)

# Slow motion on nums = [1, 2, 3, 4]:
# step   acc   x   acc*x
#   1      1   2     2
#   2      2   3     6
#   3      6   4    24   <- final result

def test_harness_reduce(fn):
    tests = [
        ([1, 2, 3, 4], 24),       # basic product
        ([5], 5),                  # single element — reduce returns it unchanged
        ([2, 2, 2], 8),            # powers of 2
        ([-1, 2, -3], 6),          # negatives cancel
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness_reduce(product_of_array)
print("product_of_array defined.")


# ── lru_cache ────────────────────────────────────────────────────────────────
# Think: sticky notes on arguments — if you've seen this input, grab the note
# maxsize=None = unlimited cache; every unique arg combo gets its own sticky note

@lru_cache(maxsize=None)
def climb_stairs(n: int) -> int:
    """
    LC 70 — Climbing Stairs.
    Approach: fib recurrence cached — @lru_cache replaces the memo dict.
    Args:
        n (int): number of stairs, n >= 1.
    Returns:
        int: number of distinct ways to reach the top.
    Time:  O(n) — each sub-problem solved once, then cached
    Space: O(n) — cache stores one entry per unique n
    """
    if n <= 2:
        return n                        # base: 1 stair=1 way, 2 stairs=2 ways
    return climb_stairs(n - 1) + climb_stairs(n - 2)  # look up both — both cached after first call

def test_harness_lru(fn):
    tests = [
        (1, 1),
        (2, 2),
        (3, 3),
        (5, 8),
        (10, 89),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness_lru(climb_stairs)
print("climb_stairs defined.")


# ── cmp_to_key ───────────────────────────────────────────────────────────────
# Think: referee — you give it a fight rule, it converts fights into scores for sorted()
# cmp(a, b) must return: negative = a wins (goes first), 0 = tie, positive = b wins

def largest_number(nums: List[int]) -> str:
    """
    LC 179 — Largest Number.
    Approach: custom comparator — compare "ab" vs "ba" as concatenated strings.
    Args:
        nums (List[int]): list of non-negative integers.
    Returns:
        str: largest number formed by concatenating all elements.
    Time:  O(n log n) — sort with O(1) comparator, O(n) comparisons
    Space: O(n) — string conversion of each number
    """
    strs = [str(n) for n in nums]        # convert to strings for concat comparison

    def compare(a, b):
        # which concatenation is bigger?
        if a + b > b + a:
            return -1                    # a should go first
        elif a + b < b + a:
            return 1                     # b should go first
        return 0

    strs.sort(key=cmp_to_key(compare))
    result = "".join(strs)
    return "0" if result[0] == "0" else result  # edge case: all zeros

# Slow motion on nums = [3, 30, 34, 5, 9]:
# compare("9", "5")  -> "95" > "59" -> -1 -> 9 goes first
# compare("34","3")  -> "343" > "334" -> -1 -> 34 goes first
# sorted: ["9","5","34","3","30"] -> "9534330"

def test_harness_largest(fn):
    tests = [
        ([10, 2], "210"),
        ([3, 30, 34, 5, 9], "9534330"),
        ([0, 0], "0"),
        ([1], "1"),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness_largest(largest_number)
print("largest_number defined.")


# ── partial() ────────────────────────────────────────────────────────────────
# Think: pre-filling a form — lock in some fields, leave others blank

def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)    # base is still free
cube   = partial(power, exponent=3)    # different pre-fill

print(square(5))    # 25  — only needed to pass base
print(cube(3))      # 27

# Practical: pre-fill a sort key
from functools import partial

def sort_by_nth(lst, n):
    return sorted(lst, key=lambda x: x[n])

sort_by_first = partial(sort_by_nth, n=0)
sort_by_second = partial(sort_by_nth, n=1)

pairs = [(3, 'c'), (1, 'a'), (2, 'b')]
print(sort_by_first(pairs))   # [(1,'a'), (2,'b'), (3,'c')]
print(sort_by_second(pairs))  # [(1,'a'), (2,'b'), (3,'c')]  same here, but pattern holds

print("partial demo complete.")

# Simplicity and clarity is Gold
